In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 509, done.
remote: Counting objects: 100% (285/285), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 509 (delta 114), reused 139 (delta 22), pack-reused 224 (from 1)
Receiving objects: 100% (509/509), 16.05 MiB | 13.04 MiB/s, done.
Resolving deltas: 100% (229/229), done.
Filtering content: 100% (58/58), 104.47 MiB | 25.38 MiB/s, done.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
import json
import os
from sklearn.model_selection import cross_val_score

In [3]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/wnba_final_df.csv")


# preview
df.head()

,team,year,home_away,opp,win_loss,team_score,opp_score,team_fg,team_fga,team_fg_pct,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
0,ATL,2020,1,DAL,W,105.0,95.0,34,62,0.548,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,ATL,2020,0,LVA,L,70.0,100.0,28,70,0.400,...,0.519987,0.488460,0.647610,0.053184,0.483734,0.139044,0.211406,0.527895,0.126215,0.832549
2,ATL,2020,1,NYL,W,84.0,78.0,28,75,0.373,...,0.525860,0.499134,0.617932,0.055769,0.480293,0.130352,0.232056,0.503690,0.167922,0.825130
3,ATL,2020,0,IND,L,77.0,93.0,33,69,0.478,...,0.565750,0.498973,0.649039,0.059870,0.518977,0.139425,0.249410,0.474175,0.159170,0.783814
4,ATL,2020,1,PHO,L,74.0,81.0,29,62,0.468,...,0.535450,0.519306,0.630835,0.052168,0.492024,0.136151,0.265374,0.503011,0.157341,0.834086


In [ ]:
df.shape

(2040, 362)

In [4]:
df.rename(columns={"score": "opp_score"}, inplace=True)

# Verify the change
print("score" in df.columns, "opp_score" in df.columns)

False True


In [5]:
# drop specified columns from ytd_df
df = df.drop(['ytd_team_pts_per_game', 'ytd_opp_pts_per_game'], axis=1)

KeyError: "['ytd_team_pts_per_game', 'ytd_opp_pts_per_game'] not found in axis"

In [10]:
# calculate rolling 10-game avg team points allowed, shifted by 1 game
df['rolling_10_team_allowed_mean'] = (
    df.groupby(['team','year'])['opp_score']
    .transform(lambda x: x.shift().rolling(10, min_periods=1).mean())
)

# quick verification print
print(df[['team', 'opp_score', 'rolling_10_team_allowed_mean']].head(10))

  team  opp_score  rolling_10_team_allowed_mean
0  ATL       95.0                           NaN
1  ATL      100.0                     95.000000
2  ATL       78.0                     97.500000
3  ATL       93.0                     91.000000
4  ATL       81.0                     91.500000
5  ATL       93.0                     89.400000
6  ATL       85.0                     90.000000
7  ATL       93.0                     89.285714
8  ATL      100.0                     89.750000
9  ATL       96.0                     90.888889


In [11]:
# calculate rolling 10-game avg team points allowed, shifted by 1 game
df['rolling_10_team_allowed_median'] = (
    df.groupby(['team','year'])['opp_score']
    .transform(lambda x: x.shift().rolling(10, min_periods=1).median())
)

In [12]:
# calculate rolling 10-game avg team points allowed, shifted by 1 game
df['rolling_10_team_allowed_max'] = (
    df.groupby(['team','year'])['opp_score']
    .transform(lambda x: x.shift().rolling(10, min_periods=1).max())
)

# quick verification print
print(df[['team', 'opp_score', 'rolling_10_allowed_max']].head(10))

  team  opp_score  ytd_team_allowed_max
0  ATL       95.0                   NaN
1  ATL      100.0                  95.0
2  ATL       78.0                 100.0
3  ATL       93.0                 100.0
4  ATL       81.0                 100.0
5  ATL       93.0                 100.0
6  ATL       85.0                 100.0
7  ATL       93.0                 100.0
8  ATL      100.0                 100.0
9  ATL       96.0                 100.0


In [14]:
# calculate rolling 10-game avg team points allowed, shifted by 1 game
df['rolling_10_team_allowed_min'] = (
    df.groupby(['team','year'])['opp_score']
    .transform(lambda x: x.shift().rolling(10, min_periods=1).min())
)
# quick verification print
print(df[['team', 'opp_score', 'rolling_10_team_allowed_min']].head(10))

  team  opp_score  rolling_10_team_allowed_min
0  ATL       95.0                          NaN
1  ATL      100.0                         95.0
2  ATL       78.0                         95.0
3  ATL       93.0                         78.0
4  ATL       81.0                         78.0
5  ATL       93.0                         78.0
6  ATL       85.0                         78.0
7  ATL       93.0                         78.0
8  ATL      100.0                         78.0
9  ATL       96.0                         78.0


In [15]:
# map ytd_team_allowed cols to ytd_opp_allowed cols
allowed_cols = [
    'rolling_10_team_allowed_mean',
    'rolling_10_team_allowed_median',
    'rolling_10_team_allowed_min',
    'rolling_10_team_allowed_max'
]

# loop to create ytd_opp_allowed columns
for col in allowed_cols:
    new_col = col.replace('_team_allowed_', '_opp_allowed_')

    def map_team_to_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(map_team_to_opp, axis=1)

# verification print
print(df[['team', 'opp', 'year', 'month', 'day'] + [c.replace('_team_allowed_', '_opp_allowed_') for c in allowed_cols]].head(15))

   team  opp  year  month   day  rolling_10_opp_allowed_mean  \
0   ATL  DAL  2020    7.0  26.0                     0.000000   
1   ATL  LVA  2020    7.0  29.0                          NaN   
2   ATL  NYL  2020    7.0  31.0                    87.000000   
3   ATL  IND  2020    8.0   2.0                   100.500000   
4   ATL  PHO  2020    8.0   4.0                   100.000000   
5   ATL  SEA  2020    8.0   6.0                    75.250000   
6   ATL  DAL  2020    8.0   8.0                    83.800000   
7   ATL  CON  2020    8.0  10.0                    80.833333   
8   ATL  SEA  2020    8.0  12.0                    76.428571   
9   ATL  PHO  2020    8.0  14.0                    85.000000   
10  ATL  CHI  2020    8.0  16.0                    84.555556   
11  ATL  WAS  2020    8.0  19.0                    80.111111   
12  ATL  LAS  2020    8.0  21.0                    78.900000   
13  ATL  MIN  2020    8.0  23.0                    76.600000   
14  ATL  MIN  2020    8.0  28.0         

In [17]:
# confirm if LVA's first rows have NaNs or zeroes
print(df[df['team'] == 'LVA'][['team', 'year', 'month', 'day', 'opp_score',
                               'rolling_10_team_allowed_mean',
                               'rolling_10_team_allowed_median',
                               'rolling_10_team_allowed_min',
                               'rolling_10_team_allowed_max']].head(10))

     team  year  month   day  opp_score  rolling_10_team_allowed_mean  \
1020  LVA  2020    7.0  26.0       88.0                           NaN   
1021  LVA  2020    7.0  29.0       70.0                     88.000000   
1022  LVA  2020    7.0  31.0      102.0                     79.000000   
1023  LVA  2020    8.0   2.0       70.0                     86.666667   
1024  LVA  2020    8.0   5.0       77.0                     82.500000   
1025  LVA  2020    8.0   7.0       82.0                     81.400000   
1026  LVA  2020    8.0   9.0       76.0                     81.500000   
1027  LVA  2020    8.0  11.0       79.0                     80.714286   
1028  LVA  2020    8.0  13.0       77.0                     80.500000   
1029  LVA  2020    8.0  15.0       73.0                     80.111111   

      rolling_10_team_allowed_median  rolling_10_team_allowed_min  \
1020                             NaN                          NaN   
1021                            88.0                      

In [23]:
# check number of nulls in ytd_opp_allowed columns
opp_allowed_cols = [col for col in df.columns if 'rolling_10_opp_allowed_' in col]
print(df[opp_allowed_cols].isnull().sum())

rolling_10_opp_allowed_fg_per_game                   0
rolling_10_opp_allowed_fga_per_game                  0
rolling_10_opp_allowed_3p_per_game                   0
rolling_10_opp_allowed_3pa_per_game                  0
rolling_10_opp_allowed_ft_per_game                   0
rolling_10_opp_allowed_fta_per_game                  0
rolling_10_opp_allowed_orb_per_game                  0
rolling_10_opp_allowed_trb_per_game                  0
rolling_10_opp_allowed_ast_per_game                  0
rolling_10_opp_allowed_stl_per_game                  0
rolling_10_opp_allowed_blk_per_game                  0
rolling_10_opp_allowed_tov_per_game                  0
rolling_10_opp_allowed_pf_per_game                   0
rolling_10_opp_allowed_advanced_stl_pct_per_game     0
rolling_10_opp_allowed_fg_pct                        0
rolling_10_opp_allowed_3p_pct                        0
rolling_10_opp_allowed_ft_pct                        0
rolling_10_opp_allowed_advanced_ts_pct               0
rolling_10

In [22]:
# check number of nulls in ytd_opp_allowed columns
team_allowed_cols = [col for col in df.columns if 'rolling_10_team_allowed_' in col]
print(df[team_allowed_cols].isnull().sum())

rolling_10_team_allowed_fg_per_game                   0
rolling_10_team_allowed_fga_per_game                  0
rolling_10_team_allowed_3p_per_game                   0
rolling_10_team_allowed_3pa_per_game                  0
rolling_10_team_allowed_ft_per_game                   0
rolling_10_team_allowed_fta_per_game                  0
rolling_10_team_allowed_orb_per_game                  0
rolling_10_team_allowed_trb_per_game                  0
rolling_10_team_allowed_ast_per_game                  0
rolling_10_team_allowed_stl_per_game                  0
rolling_10_team_allowed_blk_per_game                  0
rolling_10_team_allowed_tov_per_game                  0
rolling_10_team_allowed_pf_per_game                   0
rolling_10_team_allowed_advanced_stl_pct_per_game     0
rolling_10_team_allowed_fg_pct                        0
rolling_10_team_allowed_3p_pct                        0
rolling_10_team_allowed_ft_pct                        0
rolling_10_team_allowed_advanced_ts_pct         

In [24]:
# replace NaNs in opp allowed columns with 0
df[opp_allowed_cols] = df[opp_allowed_cols].fillna(0)

# quick verify
print(df[opp_allowed_cols].isnull().sum())

rolling_10_opp_allowed_fg_per_game                  0
rolling_10_opp_allowed_fga_per_game                 0
rolling_10_opp_allowed_3p_per_game                  0
rolling_10_opp_allowed_3pa_per_game                 0
rolling_10_opp_allowed_ft_per_game                  0
rolling_10_opp_allowed_fta_per_game                 0
rolling_10_opp_allowed_orb_per_game                 0
rolling_10_opp_allowed_trb_per_game                 0
rolling_10_opp_allowed_ast_per_game                 0
rolling_10_opp_allowed_stl_per_game                 0
rolling_10_opp_allowed_blk_per_game                 0
rolling_10_opp_allowed_tov_per_game                 0
rolling_10_opp_allowed_pf_per_game                  0
rolling_10_opp_allowed_advanced_stl_pct_per_game    0
rolling_10_opp_allowed_fg_pct                       0
rolling_10_opp_allowed_3p_pct                       0
rolling_10_opp_allowed_ft_pct                       0
rolling_10_opp_allowed_advanced_ts_pct              0
rolling_10_opp_allowed_offen

In [25]:
# replace NaNs in opp allowed columns with 0
df[team_allowed_cols] = df[team_allowed_cols].fillna(0)

# quick verify
print(df[team_allowed_cols].isnull().sum())

rolling_10_team_allowed_fg_per_game                  0
rolling_10_team_allowed_fga_per_game                 0
rolling_10_team_allowed_3p_per_game                  0
rolling_10_team_allowed_3pa_per_game                 0
rolling_10_team_allowed_ft_per_game                  0
rolling_10_team_allowed_fta_per_game                 0
rolling_10_team_allowed_orb_per_game                 0
rolling_10_team_allowed_trb_per_game                 0
rolling_10_team_allowed_ast_per_game                 0
rolling_10_team_allowed_stl_per_game                 0
rolling_10_team_allowed_blk_per_game                 0
rolling_10_team_allowed_tov_per_game                 0
rolling_10_team_allowed_pf_per_game                  0
rolling_10_team_allowed_advanced_stl_pct_per_game    0
rolling_10_team_allowed_fg_pct                       0
rolling_10_team_allowed_3p_pct                       0
rolling_10_team_allowed_ft_pct                       0
rolling_10_team_allowed_advanced_ts_pct              0
rolling_10

In [27]:
for col in df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opp_fg
opp_fga
opp_fg_pct
opp_3p
opp_3pa
opp_3p_pct
opp_ft
opp_fta
opp_ft_pct
opp_orb
opp_trb
opp_ast
opp_stl
opp_blk
opp_tov
opp_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_prot

In [29]:
# check number of nulls in ytd_opp_allowed columns
ytd_team_allowed_cols = [col for col in df.columns if 'ytd_team_allowed_' in col]
print(df[ytd_team_allowed_cols].isnull().sum())

ytd_team_allowed_mean                         12
ytd_team_allowed_median                       12
ytd_team_allowed_min                          12
ytd_team_allowed_max                          12
ytd_team_allowed_fg_per_game                   0
ytd_team_allowed_fga_per_game                  0
ytd_team_allowed_3p_per_game                   0
ytd_team_allowed_3pa_per_game                  0
ytd_team_allowed_ft_per_game                   0
ytd_team_allowed_fta_per_game                  0
ytd_team_allowed_orb_per_game                  0
ytd_team_allowed_trb_per_game                  0
ytd_team_allowed_ast_per_game                  0
ytd_team_allowed_stl_per_game                  0
ytd_team_allowed_blk_per_game                  0
ytd_team_allowed_tov_per_game                  0
ytd_team_allowed_pf_per_game                   0
ytd_team_allowed_advanced_stl_pct_per_game     0
ytd_team_allowed_fg_pct                        0
ytd_team_allowed_3p_pct                        0
ytd_team_allowed_ft_

In [30]:
# replace NaNs in opp allowed columns with 0
df[ytd_team_allowed_cols] = df[ytd_team_allowed_cols].fillna(0)

# quick verify
print(df[ytd_team_allowed_cols].isnull().sum())

ytd_team_allowed_mean                         0
ytd_team_allowed_median                       0
ytd_team_allowed_min                          0
ytd_team_allowed_max                          0
ytd_team_allowed_fg_per_game                  0
ytd_team_allowed_fga_per_game                 0
ytd_team_allowed_3p_per_game                  0
ytd_team_allowed_3pa_per_game                 0
ytd_team_allowed_ft_per_game                  0
ytd_team_allowed_fta_per_game                 0
ytd_team_allowed_orb_per_game                 0
ytd_team_allowed_trb_per_game                 0
ytd_team_allowed_ast_per_game                 0
ytd_team_allowed_stl_per_game                 0
ytd_team_allowed_blk_per_game                 0
ytd_team_allowed_tov_per_game                 0
ytd_team_allowed_pf_per_game                  0
ytd_team_allowed_advanced_stl_pct_per_game    0
ytd_team_allowed_fg_pct                       0
ytd_team_allowed_3p_pct                       0
ytd_team_allowed_ft_pct                 

In [31]:
# display columns with null counts sorted descending
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].sort_values(ascending=False))

Series([], dtype: int64)


In [41]:
# Save DataFrame as CSV file locally in Colab
df.to_csv('wnba_final_df.csv', index=False)

# Download directly to your local machine
from google.colab import files
files.download('wnba_final_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
# columns to drop from df to create df_model
drop_cols = [
    'team_fg', 'team_fga', 'team_fg_pct', 'team_3p', 'team_3pa', 'team_3p_pct',
    'team_ft', 'team_fta', 'team_ft_pct', 'team_orb', 'team_trb', 'team_ast',
    'team_stl', 'team_blk', 'team_tov', 'team_pf', 'opp_fg', 'opp_fga',
    'opp_fg_pct', 'opp_3p', 'opp_3pa', 'opp_3p_pct', 'opp_ft', 'opp_fta',
    'opp_ft_pct', 'opp_orb', 'opp_trb', 'opp_ast', 'opp_stl', 'opp_blk',
    'opp_tov', 'opp_pf', 'win_loss', 'game_num'
]

# create df_model explicitly
df_model = df.drop(columns=drop_cols)

# verify dropped columns
for col in df_model.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
ytd_team_score_mean
ytd_team_score_median
ytd_team_score_min
ytd_team_score_max
ytd_opp_score_mean
ytd_opp_score_median
ytd_opp_score_min
ytd_opp_score_max
ytd_team_allowed_mean
ytd_team

In [34]:
# columns to drop from df to create df_model
drop_cols = [
    'advanced_ortg', 'advanced_drtg', 'advanced_pace', 'advanced_ftr',
    'advanced_3par', 'advanced_ts_pct', 'advanced_trb_pct', 'advanced_ast_pct',
    'advanced_stl_pct', 'advanced_blk_pct', 'offensive_four_factors_efg_pct',
    'offensive_four_factors_tov_pct', 'offensive_four_factors_orb_pct',
    'defensive_four_factors_efg_pct', 'defensive_four_factors_tov_pct',
    'defensive_four_factors_drb_pct'
]

# create df_model explicitly
df_model = df_model.drop(columns=drop_cols)

# verify dropped columns
for col in df_model.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
ytd_team_score_mean
ytd_team_score_median
ytd_team_score_min
ytd_team_score_max
ytd_opp_score_mean
ytd_opp_score_median
ytd_opp_score_min
ytd_opp_score_max
ytd_team_allowed_mean
ytd_team_allowed_median
ytd_team_allowed_min
ytd_team_allowed_max
ytd_opp_allowed_mean
ytd_opp_allowed_median
ytd_opp_allowed_min
ytd_opp_allowed_max
ytd_team_fg_per_game
ytd_team_fga_per_game
ytd_team_3p_per_game
ytd_team_3pa_per_game
ytd_team_ft_per_game
ytd_team_fta_per_game
ytd_team_orb_per_game
ytd_team_trb_per_game
ytd_team_ast_per_game
yt

In [35]:
# Get numeric columns only
numeric_cols = df_model.select_dtypes(include=[np.number]).columns

# Check infinite values explicitly in numeric columns
inf_cols_df_model = numeric_cols[np.isinf(df_model[numeric_cols]).any()]
print("Numeric columns with infinite values:", inf_cols_df_model.tolist())

# Count infinite values explicitly per column
inf_counts = df_model[inf_cols_df_model].apply(lambda col: np.isinf(col).sum())
print(inf_counts)

Numeric columns with infinite values: ['rolling_10_team_advanced_blk_pct', 'rolling_10_opp_advanced_blk_pct']
rolling_10_team_advanced_blk_pct    59
rolling_10_opp_advanced_blk_pct     59
dtype: int64


In [34]:
# Check original columns involved in blk_pct calculation for rows with infinite blk_pct
check_cols = [
    'team', 'year', 'rolling_10_team_blk_per_game', 'rolling_10_opp_fga_per_game',
    'rolling_10_opp_blk_per_game', 'rolling_10_team_fga_per_game',
    'rolling_10_team_advanced_blk_pct', 'rolling_10_opp_advanced_blk_pct'
]

inf_rows = df_model[
    np.isinf(df_model["rolling_10_team_advanced_blk_pct"]) |
    np.isinf(df_model["rolling_10_opp_advanced_blk_pct"])
][check_cols]

print(inf_rows.head(60))

     team  year  rolling_10_team_blk_per_game  rolling_10_opp_fga_per_game  \
1     ATL  2020                      4.000000                     0.000000   
2     ATL  2020                      4.500000                    66.000000   
23    ATL  2021                      7.000000                     0.000000   
55    ATL  2022                      9.000000                    65.000000   
91    ATL  2023                      6.000000                     0.000000   
92    ATL  2023                      6.000000                    66.000000   
131   ATL  2024                      9.000000                     0.000000   
132   ATL  2024                      7.000000                    82.000000   
171   CHI  2020                      3.000000                     0.000000   
172   CHI  2020                      2.500000                    67.000000   
193   CHI  2021                      3.000000                     0.000000   
225   CHI  2022                      2.000000                   

In [36]:
# explicitly drop rows where rolling_10_team_fga_per_game is exactly zero
df_model = df_model[df_model["rolling_10_team_fga_per_game"] != 0].copy()

# explicitly replace all remaining infinite values with zero
df_model.replace([np.inf, -np.inf], 0, inplace=True)

# verification
print("Remaining infinite values:", np.isinf(df_model.select_dtypes(include=[np.number])).sum().sum())
print("Rows with rolling_10_team_fga_per_game = 0:", (df_model["rolling_10_team_fga_per_game"] == 0).sum())

Remaining infinite values: 0
Rows with rolling_10_team_fga_per_game = 0: 0


In [37]:
# explicitly define leakage-prevention logic
year_cols = {
    2023: ['2023'],
    2022: ['2022', '2023'],
    2021: ['2021', '2022', '2023'],
    2020: ['2020', '2021', '2022', '2023']
}

# loop clearly to zero future values to prevent leakage
for yr, yrs_to_zero in year_cols.items():
    for col_year in yrs_to_zero:
        cols_to_zero = [col for col in df_model.columns if col_year in col]
        df_model.loc[df_model['year'] == yr, cols_to_zero] = 0

# quick explicit verification
for yr in [2020, 2021, 2022, 2023]:
    cols_checked = [col for col in df_model.columns if any(str(y) in col for y in range(yr, 2024))]
    print(f"Year {yr}, sum after zeroing:", df_model.loc[df_model["year"] == yr, cols_checked].sum().sum())

Year 2020, sum after zeroing: 0.0
Year 2021, sum after zeroing: 0.0
Year 2022, sum after zeroing: 0.0
Year 2023, sum after zeroing: 0.0


In [11]:
# explicitly define years and sample teams for spot-checking
sample_years = [2020, 2021, 2022, 2023]
sample_cols = {yr: [col for col in df_model.columns if any(str(y) in col for y in range(yr, 2024))] for yr in sample_years}

# print a few spot-check rows per year explicitly
for yr in sample_years:
    print(f"\n=== Year: {yr} ===")
    display(df_model[df_model['year'] == yr][['team', 'year'] + sample_cols[yr]].sample(3, random_state=42))


=== Year: 2020 ===


,team,year,2023_team_fg_per_game,2023_team_fga_per_game,2023_team_3p_per_game,2023_team_3pa_per_game,2023_team_ft_per_game,2023_team_fta_per_game,2023_team_orb_per_game,2023_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
1209,MIN,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,ATL,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
857,LAS,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Year: 2021 ===


,team,year,2023_team_fg_per_game,2023_team_fga_per_game,2023_team_3p_per_game,2023_team_3pa_per_game,2023_team_ft_per_game,2023_team_fta_per_game,2023_team_orb_per_game,2023_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
1740,SEA,2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
195,CHI,2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38,ATL,2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Year: 2022 ===


,team,year,2023_team_fg_per_game,2023_team_fga_per_game,2023_team_3p_per_game,2023_team_3pa_per_game,2023_team_ft_per_game,2023_team_fta_per_game,2023_team_orb_per_game,2023_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
740,IND,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1604,PHO,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
905,LAS,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Year: 2023 ===


,team,year,2023_team_fg_per_game,2023_team_fga_per_game,2023_team_3p_per_game,2023_team_3pa_per_game,2023_team_ft_per_game,2023_team_fta_per_game,2023_team_orb_per_game,2023_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
277,CHI,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285,CHI,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
124,ATL,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# explicitly define years to check and columns that SHOULD NOT be zero
check_years = {
    2023: ['2020', '2021', '2022'],
    2022: ['2020', '2021'],
    2021: ['2020']
}

# explicitly print rows to verify non-zero values
for yr, valid_years in check_years.items():
    cols_not_zero = [col for col in df_model.columns if any(y in col for y in valid_years)]
    print(f"\n=== Year: {yr} (columns should NOT be zero) ===")
    display(df_model[df_model['year'] == yr][['team', 'year'] + cols_not_zero].sample(3, random_state=42))


=== Year: 2023 (columns should NOT be zero) ===


,team,year,2022_team_fg_per_game,2022_team_fga_per_game,2022_team_3p_per_game,2022_team_3pa_per_game,2022_team_ft_per_game,2022_team_fta_per_game,2022_team_orb_per_game,2022_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
277,CHI,2023,32.666667,67.861111,7.222222,20.944444,13.694444,16.638889,7.222222,34.833333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285,CHI,2023,32.666667,67.861111,7.222222,20.944444,13.694444,16.638889,7.222222,34.833333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
124,ATL,2023,28.750000,68.416667,7.527778,21.416667,13.472222,17.333333,8.611111,35.444444,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Year: 2022 (columns should NOT be zero) ===


,team,year,2021_team_fg_per_game,2021_team_fga_per_game,2021_team_3p_per_game,2021_team_3pa_per_game,2021_team_ft_per_game,2021_team_fta_per_game,2021_team_orb_per_game,2021_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
740,IND,2022,28.40625,68.1875,4.90625,17.25000,13.59375,17.00000,9.59375,34.34375,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1604,PHO,2022,29.53125,65.6875,7.53125,21.78125,15.46875,19.37500,7.96875,36.18750,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
905,LAS,2022,27.03125,65.8125,6.75000,20.12500,11.93750,15.59375,6.09375,29.25000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Year: 2021 (columns should NOT be zero) ===


,team,year,2020_team_fg_per_game,2020_team_fga_per_game,2020_team_3p_per_game,2020_team_3pa_per_game,2020_team_ft_per_game,2020_team_fta_per_game,2020_team_orb_per_game,2020_team_trb_per_game,...,2020_2021_2022_2023_opp_allowed_advanced_ts_pct,2020_2021_2022_2023_opp_allowed_advanced_trb_pct,2020_2021_2022_2023_opp_allowed_advanced_ast_pct,2020_2021_2022_2023_opp_allowed_advanced_blk_pct,2020_2021_2022_2023_opp_allowed_offensive_efg_pct,2020_2021_2022_2023_opp_allowed_offensive_tov_pct,2020_2021_2022_2023_opp_allowed_offensive_orb_pct,2020_2021_2022_2023_opp_allowed_defensive_efg_pct,2020_2021_2022_2023_opp_allowed_defensive_tov_pct,2020_2021_2022_2023_opp_allowed_defensive_drb_pct
1740,SEA,2021,31.863636,67.772727,8.409091,21.363636,15.409091,18.863636,7.590909,34.409091,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
195,CHI,2021,33.500000,68.181818,7.590909,21.636364,12.136364,14.863636,7.318182,33.590909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38,ATL,2021,31.409091,71.045455,5.909091,16.863636,12.227273,16.136364,9.045455,34.863636,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
# Save DataFrame as CSV file locally in Colab
df_model.to_csv('wnba_model_df.csv', index=False)

# Download directly to your local machine
from google.colab import files
files.download('wnba_model_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
team_name = "League"
model_df = df_model

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# initialize basic XGB model (standard params)
model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=1,
    colsample_bytree=1,
    random_state=42,
    verbosity=0
)

# train model
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,5.54192,4.647259,0.587248,8.545823,7.316578,0.033304


In [25]:
from sklearn.model_selection import GridSearchCV

# parameter grid for quick optimization
param_grid = {
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1],
    'colsample_bytree': [0.6, 0.8, 1],
    'n_estimators': [100, 300, 500]
}

# initialize XGBRegressor
xgb = XGBRegressor(random_state=42, verbosity=0)

# setup grid search with 5-fold CV
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

# run grid search
grid_search.fit(X_train, y_train)

# best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best MAE (CV):", -grid_search.best_score_)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.6}
Best MAE (CV): 8.292834432235617


In [38]:
# explicitly identify rolling_10 and ytd columns
rolling_10_cols = [col for col in df_model.columns if 'rolling_10' in col]
ytd_cols = [col for col in df_model.columns if 'ytd' in col]

# create rolling_10_df (drop ytd columns)
rolling_10_df = df_model.drop(columns=ytd_cols).copy()

# create ytd_df (drop rolling_10 columns)
ytd_df = df_model.drop(columns=rolling_10_cols).copy()

# verification
print("rolling_10_df shape:", rolling_10_df.shape)
print("ytd_df shape:", ytd_df.shape)
print("df_model shape:", df_model.shape)

rolling_10_df shape: (1980, 1410)
ytd_df shape: (1980, 1410)
df_model shape: (1980, 1538)


In [40]:
# columns to rename
cols_to_rename = [
    'all_around_star.1', 'and_one_machine.1', 'catch_and_shoot.1', 'corner_3_specialist.1',
    'defensive_anchor.1', 'defensive_rebounder.1', 'efficient_scorer.1', 'elite_scorer.1',
    'fast_break_threat.1', 'floor_general.1', 'free_throw_generator.1', 'glass_cleaner.1',
    'heave_chucker.1', 'impact_bench.1', 'midrange_sniper.1', 'offensive_hub.1',
    'offensive_rebounder.1', 'playmaker.1', 'plus_minus_driver.1', 'rim_protector.1',
    'self_creator.1', 'slasher.1', 'steal_artist.1', 'stretch_big.1',
    'three_point_specialist.1', 'turnover_prone.1', 'volume_shooter.1'
]

# create rename mapping dictionary
rename_mapping = {col: 'opp_' + col.replace('.1', '') for col in cols_to_rename}

# apply rename to all dfs
for current_df in [df, df_model, ytd_df, rolling_10_df]:
    current_df.rename(columns=rename_mapping, inplace=True)

# verification print for df
print(df[list(rename_mapping.values())].head(5))

   opp_all_around_star  opp_and_one_machine  opp_catch_and_shoot  \
0                  0.0                  1.0                  0.0   
1                  0.0                  0.0                  0.0   
2                  0.0                  0.0                  0.0   
3                  0.0                  0.0                  0.0   
4                  0.0                  0.0                  0.0   

   opp_corner_3_specialist  opp_defensive_anchor  opp_defensive_rebounder  \
0                      0.0                   0.0                      1.0   
1                      0.0                   0.0                      0.0   
2                      0.0                   0.0                      0.0   
3                      0.0                   0.0                      0.0   
4                      0.0                   0.0                      0.0   

   opp_efficient_scorer  opp_elite_scorer  opp_fast_break_threat  \
0                   0.0               0.0                   

In [43]:
for col in rolling_10_df.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
rolling_10_team_score_mean
rolling_10_team_score_median
rolling_10_team_score_min
rolling_10_team_score_max
rolling_10_opp_score_mean
rolling_10_opp_score_median
rolling_10_opp_score_min
rolling_10_opp_score_max
rolling_10_team_fg_per_game
rolling_10_team_fga_per_game
rolling_10_team_3p_per_game
rolling_10_team_3pa_per_game
rolling_10_team_ft_per_game
rolling_10_team_fta_per_game
rolling_10_team_orb_per_game
rolling_10_team_trb_per_game
rolling_10_team_ast_per_game
rolling_10_team_stl_per_game
rolling_10_team_blk_per_ga

In [40]:
for col in ytd_df.columns:
  print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
ytd_team_score_mean
ytd_team_score_median
ytd_team_score_min
ytd_team_score_max
ytd_opp_score_mean
ytd_opp_score_median
ytd_opp_score_min
ytd_opp_score_max
ytd_team_allowed_mean
ytd_team_allowed_median
ytd_team_allowed_min
ytd_team_allowed_max
ytd_opp_allowed_mean
ytd_opp_allowed_median
ytd_opp_allowed_min
ytd_opp_allowed_max
ytd_team_fg_per_game
ytd_team_fga_per_game
ytd_team_3p_per_game
ytd_team_3pa_per_game
ytd_team_ft_per_game
ytd_team_fta_per_game
ytd_team_orb_per_game
ytd_team_trb_per_game
ytd_team_ast_per_game
yt

In [44]:
# Save DataFrame as CSV file locally in Colab
rolling_10_df.to_csv('rolling_10_model_df.csv', index=False)

# Download directly to your local machine
from google.colab import files
files.download('rolling_10_model_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
# Save DataFrame as CSV file locally in Colab
ytd_df.to_csv('ytd_model_df.csv', index=False)

# Download directly to your local machine
from google.colab import files
files.download('ytd_model_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>